# Mastery Workbook: solutions

One correct answer per exercise. Try first; after reading a solution, close it and write it again from memory.

The first code cell holds small helpers the solutions reuse (reading every Stringee file, the gap calculation, and so on).

In [ ]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

from practice_data import load, sql, build
from checker import check, progress
from solutions import *  # helpers + constants

D = load()               # first run builds the practice files (a few seconds)
P = D['paths']           # every folder and file: P['stringee_dir'], P['ledger_csv'], ...
S = D['strings']         # text practice table for module 3
pd.set_option('display.max_columns', None)
print('Practice files in:', P['root'])
print('Tables in D:', [k for k, v in D.items() if isinstance(v, pd.DataFrame)])

## Helpers used by several solutions

In [ ]:
def ref_date_range_name(prefix, dates, ext):
    first, last = dates.min(), dates.max()
    if first.date() == last.date():
        return f"{prefix}_{first:%Y-%m-%d}{ext}"
    return f"{prefix}_{first:%Y-%m-%d}_to_{last:%Y-%m-%d}{ext}"


def ref_norm(name):
    return re.sub(r"[^a-z0-9]", "", str(name).lower())


def ref_clean_dialer_name(s):
    s = s.astype("string").str.replace(r"\s*\([^)]*\)|@.*|;.*", "", regex=True)
    return (s.str.replace(r"\s+", " ", regex=True).str.strip()
             .str.replace(r"@.*", "", regex=True).str.replace(r"\(.*", "", regex=True)
             .str.strip().str.lower())


def ref_to_td(s):
    txt = s.astype("string").str.strip()
    is_secs = txt.str.fullmatch(r"\d+").fillna(False).astype(bool)
    out = pd.to_timedelta(txt.mask(is_secs), errors="coerce")
    if is_secs.any():
        out[is_secs] = pd.to_timedelta(txt[is_secs].astype("int64"), unit="s")
    return out


def ref_hms(seconds):
    s = pd.Series(seconds).fillna(0).astype("int64")
    h, m, sec = s // 3600, (s % 3600) // 60, s % 60
    return h.astype(str).str.zfill(2) + ":" + m.astype(str).str.zfill(2) + ":" + sec.astype(str).str.zfill(2)


def _st_files(D):
    return sorted(f for f in D["paths"]["stringee_dir"].glob("*.xlsx") if not f.name.startswith("~$"))


def _stringee(D, source=False):
    parts = []
    for f in _st_files(D):
        p = pd.read_excel(f, dtype={"Account": str, "Customer number": str})
        if source:
            p["source_file"] = f.name
        parts.append(p)
    st = pd.concat(parts, ignore_index=True)
    st["Start time"] = pd.to_datetime(st["Start time"])
    return st


def _latest_balance(D):
    led = D["ledger"]
    return (led.sort_values("CreatedAt", ascending=False)
               .drop_duplicates(subset="ClientID", keep="first")[["ClientID", "Balance"]]
               .reset_index(drop=True))


def _gap_parts(D):
    calls = D["calls"]
    g = calls.groupby(["Date", "CRM ID"], sort=False)
    prev_start = g["Call Start Time"].shift()
    prev_end = prev_start + g["Total Call Duration"].shift()
    gap = (calls["Call Start Time"] - prev_end).clip(lower=pd.Timedelta(0))
    return calls, prev_start, gap


def _gap_seconds(D):
    calls, prev_start, gap = _gap_parts(D)
    return np.floor(gap.dt.total_seconds().fillna(0)).astype("int64")


def _slots(D):
    hours = D["calls"]["Call Start Time"].dt.hour
    idx = (hours - 7).clip(0, len(D["SLOT_LABELS"]) - 1).astype(int)
    return pd.Series(np.array(D["SLOT_LABELS"])[idx], index=D["calls"].index)


def _bill_days_pivot(D, clients=None):
    bills = D["ledger"][D["ledger"]["LedgerType"] == "BILL"].copy()
    if clients is not None:
        bills = bills[bills["ClientID"].isin(clients)]
    bills["Month"] = bills["CreatedAt"].dt.to_period("M").astype(str)
    bills["Trade Date"] = bills["CreatedAt"].dt.normalize()
    pivot = bills.pivot_table(index="ClientID", columns="Month", values="Trade Date", aggfunc="nunique", fill_value=0)
    pivot.columns.name = None
    return pivot


def _uno(D, files=("uno_2026-10-28.csv", "uno_2026-10-29.csv")):
    return pd.concat([pd.read_csv(D["paths"]["uno_dir"] / f, dtype={"Caller Number": str}) for f in files],
                     ignore_index=True)


def _talk_s(D):
    return np.floor(D["calls"]["Talk Time"].dt.total_seconds().fillna(0)).astype("int64")


def _align_map(ark_cols, main_cols):
    lookup = {ref_norm(c): c for c in main_cols}
    out = {}
    for c in ark_cols:
        new = MANUAL[c] if c in MANUAL else lookup.get(ref_norm(c))
        if new is not None:
            out[c] = new
    return out


def _gap_table(D):
    c = D["calls"].assign(gap_s=_gap_seconds(D))
    g = c.groupby(["CRM ID", "Date"], as_index=False)["gap_s"].sum()
    g["gap_s"] = np.where(g["gap_s"] > 3600, g["gap_s"] - 3600, g["gap_s"])
    return g


def _login_table(D):
    c = D["calls"].assign(total_s=np.floor(D["calls"]["Total Call Duration"].dt.total_seconds().fillna(0)).astype("int64"))
    t = c.groupby(["CRM ID", "Date"], as_index=False)["total_s"].sum()
    t = t.merge(_gap_table(D), on=["CRM ID", "Date"])
    t["login_s"] = t["total_s"] + t["gap_s"] + 3600
    return t[["CRM ID", "Date", "login_s"]]


def _lf(D):
    import polars as pl
    return pl.concat([pl.scan_csv(f, schema_overrides={"expiryDate": pl.String, "comment": pl.String})
                      for f in sorted(D["paths"]["tx_dir"].glob("*.csv"))], how="diagonal_relaxed")


def _sql(q):
    from practice_data import sql
    return sql(q)


MANUAL = {'Account': 'accountId', 'Open Com': 'openCommission', 'Close Com': 'closeCommission'}
DAMAGED = '\\d{1,2}:\\d{2}(\\.\\d+)?'
RANKED = '''
WITH ranked AS (
    SELECT cl.lead_id, cl.id AS call_id, cl.started_at, d.name AS disposition,
           ROW_NUMBER() OVER (PARTITION BY cl.lead_id ORDER BY cl.started_at DESC, cl.id DESC) AS rn
    FROM call_logs cl
    JOIN lead_calls lc ON lc.call_log_id = cl.id
    JOIN lead_call_dispositions d ON d.id = lc.disposition_id
    WHERE cl.status NOT IN ('placing', 'call_failed')
)
'''
DEDUP = '''
WITH dedup AS (
    SELECT DISTINCT ON (customer_id, occurred_at) customer_id, occurred_at, new_status
    FROM customer_status_history
    ORDER BY customer_id, occurred_at, created_at DESC, id DESC
)
'''

## Files and paths

### Exercise 1. List the Stringee exports

`ans` = a **sorted list of file names** of every `.xlsx` in `P['stringee_dir']`,
skipping Excel lock files (names starting with `~$`).

In [ ]:
def ex_001(D):
    P = D["paths"]
    return sorted(f.name for f in P["stringee_dir"].glob("*.xlsx") if not f.name.startswith("~$"))

ans = ex_001(D)
check(1, ans)
ans

### Exercise 2. List the Uno exports

`ans` = a sorted list of the `.csv` file names in `P['uno_dir']`.

In [ ]:
def ex_002(D):
    return sorted(f.name for f in D["paths"]["uno_dir"].glob("*.csv"))

ans = ex_002(D)
check(2, ans)
ans

### Exercise 3. A file check that explains itself

Write `need_files(folder, pattern, n)`: return the **sorted list of matching paths**
(skip `~$` files). If the count isn't `n`, raise `ValueError` with a message that lists the names found.

In [ ]:
def _ref_need_files(folder, pattern, n):
    files = sorted(f for f in folder.glob(pattern) if not f.name.startswith("~$"))
    if len(files) != n:
        raise ValueError(f"Expected {n} files, found {len(files)}: {[f.name for f in files]}")
    return files


check(3, _ref_need_files)

### Exercise 4. The newest team list

`ans` = the **name** of the most recently modified `.csv` or `.xlsx` in `P['team_dir']`
(use `f.stat().st_mtime`).

In [ ]:
def ex_004(D):
    files = [f for f in D["paths"]["team_dir"].iterdir() if f.suffix.lower() in (".csv", ".xlsx")]
    return max(files, key=lambda f: f.stat().st_mtime).name

ans = ex_004(D)
check(4, ans)
ans

### Exercise 5. File names with a date range

Write `date_range_name(prefix, dates, ext)` where `dates` is a datetime Series.
One calendar date → `'Summary_2026-10-28.csv'`; several → `'Summary_2026-10-28_to_2026-10-30.csv'`.

In [ ]:
def ref_date_range_name(prefix, dates, ext):
    first, last = dates.min(), dates.max()
    if first.date() == last.date():
        return f"{prefix}_{first:%Y-%m-%d}{ext}"
    return f"{prefix}_{first:%Y-%m-%d}_to_{last:%Y-%m-%d}{ext}"


check(5, ref_date_range_name)

### Exercise 6. Read whatever format the file is

Write `read_any(path)`: `pd.read_csv` for `.csv`, `pd.read_excel` for `.xlsx`,
and raise `ValueError` for anything else.

In [ ]:
def _ref_read_any(path):
    path = __import__("pathlib").Path(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() == ".xlsx":
        return pd.read_excel(path)
    raise ValueError(f"Not a .csv or .xlsx file: {path.name}")


check(6, _ref_read_any)

### Exercise 7. Pieces of a path

For `p = P['ledger_csv']`, `ans` = a tuple `(parent folder name, file stem, suffix)`.

In [ ]:
def ex_007(D):
    p = D["paths"]["ledger_csv"]
    return (p.parent.name, p.stem, p.suffix)

ans = ex_007(D)
check(7, ans)
ans

### Exercise 8. Name the processed file

Combine every Stringee export, parse `Start time`, and build the processed file name with
your `date_range_name` (prefix `'st_daily'`, ext `'.xlsx'`). `ans` = the name.

In [ ]:
def ex_008(D):
    return ref_date_range_name("st_daily", _stringee(D)["Start time"], ".xlsx")

ans = ex_008(D)
check(8, ans)
ans

### Exercise 9. Rows per export

`ans` = a dict `{file name: number of rows}` for every Stringee export.

In [ ]:
def ex_009(D):
    return {f.name: len(pd.read_excel(f)) for f in _st_files(D)}

ans = ex_009(D)
check(9, ans)
ans

### Exercise 10. Date range per export

`ans` = a dict `{file name: ('first date', 'last date')}` using each export's `Start time`,
dates as `'YYYY-MM-DD'` strings. (This is the check that found the wrong-folder problem.)

In [ ]:
def ex_010(D):
    out = {}
    for f in _st_files(D):
        t = pd.to_datetime(pd.read_excel(f)["Start time"])
        out[f.name] = (f"{t.min():%Y-%m-%d}", f"{t.max():%Y-%m-%d}")
    return out

ans = ex_010(D)
check(10, ans)
ans

## Reading and types

### Exercise 11. Phone numbers as text

Read `stringee_2026-10-28_a.xlsx` with `dtype={'Customer number': str}`.
`ans` = the first 3 values of `Customer number` as a list.

In [ ]:
def ex_011(D):
    df = pd.read_excel(D["paths"]["stringee_dir"] / "stringee_2026-10-28_a.xlsx", dtype={"Customer number": str})
    return df["Customer number"].head(3).tolist()

ans = ex_011(D)
check(11, ans)
ans

### Exercise 12. Spot the float damage

Read the same file **without** `dtype`, then `.astype(str)` the `Customer number` column.
`ans` = how many values end with `'.0'` (blanks forced the column to floats).

In [ ]:
def ex_012(D):
    df = pd.read_excel(D["paths"]["stringee_dir"] / "stringee_2026-10-28_a.xlsx")
    return int(df["Customer number"].astype(str).str.endswith(".0").sum())

ans = ex_012(D)
check(12, ans)
ans

### Exercise 13. Month-first text dates

`P['us_times_csv']` has `Start time` like `'10/29/2026 09:34:25 PM'` and two unreadable values.
Parse with an explicit format and `errors='coerce'`. `ans` = the earliest timestamp.

In [ ]:
def ex_013(D):
    df = pd.read_csv(D["paths"]["us_times_csv"])
    return pd.to_datetime(df["Start time"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce").min()

ans = ex_013(D)
check(13, ans)
ans

### Exercise 14. Count what failed

Same file and parsing. `ans` = how many values became `NaT`.

In [ ]:
def ex_014(D):
    df = pd.read_csv(D["paths"]["us_times_csv"])
    return int(pd.to_datetime(df["Start time"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce").isna().sum())

ans = ex_014(D)
check(14, ans)
ans

### Exercise 15. Uno start times

Read the two daily Uno CSVs (`uno_2026-10-28.csv`, `uno_2026-10-29.csv`). Build each start time from
`Call Date` (`'28/Oct/2026'`) and `Call Time` (`'11:58:14 pm'`). `ans` = the latest start time.

In [ ]:
def ex_015(D):
    uno = _uno(D)
    start = pd.to_datetime(pd.to_datetime(uno["Call Date"], format="%d/%b/%Y").dt.strftime("%Y-%m-%d") + " "
                           + uno["Call Time"].str.upper(), format="%Y-%m-%d %I:%M:%S %p")
    return start.max()

ans = ex_015(D)
check(15, ans)
ans

### Exercise 16. Total talk time

Combine every Stringee export (all rows), convert `Answer duration` to timedelta.
`ans` = the total in whole seconds (an int).

In [ ]:
def ex_016(D):
    return int(pd.to_timedelta(_stringee(D)["Answer duration"].astype(str)).dt.total_seconds().sum())

ans = ex_016(D)
check(16, ans)
ans

### Exercise 17. A forgiving duration parser

Write `to_td(s)` for a Series of text: `'HH:MM:SS'` (spaces allowed around it), whole seconds like
`'75'`, or blanks. Return timedeltas, with blanks as `NaT`.

In [ ]:
def ref_to_td(s):
    txt = s.astype("string").str.strip()
    is_secs = txt.str.fullmatch(r"\d+").fillna(False).astype(bool)
    out = pd.to_timedelta(txt.mask(is_secs), errors="coerce")
    if is_secs.any():
        out[is_secs] = pd.to_timedelta(txt[is_secs].astype("int64"), unit="s")
    return out


check(17, ref_to_td)

### Exercise 18. Caller numbers as text

Read `uno_2026-10-28.csv` with `Caller Number` as text. `ans` = the number of distinct caller numbers.

In [ ]:
def ex_018(D):
    df = pd.read_csv(D["paths"]["uno_dir"] / "uno_2026-10-28.csv", dtype={"Caller Number": str})
    return int(df["Caller Number"].nunique())

ans = ex_018(D)
check(18, ans)
ans

### Exercise 19. See a mixed column

Read `tx_1.csv` with default settings. `ans` = a dict `{type name: count}` for the values in `expiryDate`,
using `.map(lambda v: type(v).__name__).value_counts()`.

In [ ]:
def ex_019(D):
    df = pd.read_csv(D["paths"]["tx_dir"] / "tx_1.csv")
    return df["expiryDate"].map(lambda v: type(v).__name__).value_counts().to_dict()

ans = ex_019(D)
check(19, ans)
ans

### Exercise 20. Whole numbers with blanks

`ans` = `D['ark']['Ticket Id']` converted to pandas' nullable integer type `Int64`
(blanks stay blank; no `.0`).

In [ ]:
def ex_020(D):
    return pd.to_numeric(D["ark"]["Ticket Id"], errors="coerce").astype("Int64")

ans = ex_020(D)
check(20, ans)
ans

### Exercise 21. Read only what you need

Read `tx_1.csv` with `usecols=['ticketId', 'time', 'accountId']`. `ans` = the list of columns you got.

In [ ]:
def ex_021(D):
    return pd.read_csv(D["paths"]["tx_dir"] / "tx_1.csv", usecols=["ticketId", "time", "accountId"]).columns.tolist()

ans = ex_021(D)
check(21, ans)
ans

### Exercise 22. How big is it?

Read all of `tx_1.csv`. `ans` = its memory use in MB from `memory_usage(deep=True).sum() / 1e6`,
rounded to 2 decimals.

In [ ]:
def ex_022(D):
    df = pd.read_csv(D["paths"]["tx_dir"] / "tx_1.csv")
    return round(df.memory_usage(deep=True).sum() / 1e6, 2)

ans = ex_022(D)
check(22, ans)
ans

## Strings and regex

### Exercise 23. The second caller ID

`S = D['strings']`. `ans` = a Series: the part after `|` in `S['caller_ids']`, with spaces stripped.

In [ ]:
def ex_023(D):
    return D["strings"]["caller_ids"].str.split("|").str[1].str.strip()

ans = ex_023(D)
check(23, ans)
ans

### Exercise 24. Customer from Contact Info

`ans` = a Series of the digits after `customer :` in `S['contact_info']`.

In [ ]:
def ex_024(D):
    return D["strings"]["contact_info"].str.extract(r"customer\s*:\s*(\d+)", expand=False)

ans = ex_024(D)
check(24, ans)
ans

### Exercise 25. Agent id, not agentType

`ans` = a Series of the digits after `agent_id :` (the `agentType : agent_id` part must not match).

In [ ]:
def ex_025(D):
    return D["strings"]["contact_info"].str.extract(r"\bagent_id\s*:\s*(\d+)", expand=False)

ans = ex_025(D)
check(25, ans)
ans

### Exercise 26. Drop the country code

`ans` = the customer numbers from exercise 24, cut to their last 10 digits.

In [ ]:
def ex_026(D):
    return ex_024(D).str[-10:]

ans = ex_026(D)
check(26, ans)
ans

### Exercise 27. Digits out of an ID

`ans` = a Series of the digits in `S['br_id']`, e.g. `'BR575346: ****'` → `'575346'`.

In [ ]:
def ex_027(D):
    return D["strings"]["br_id"].str.extract(r"(\d+)", expand=False)

ans = ex_027(D)
check(27, ans)
ans

### Exercise 28. Remove a masked suffix

`ans` = `S['account_id']` without the trailing `': ****'`, e.g. `'CO657078: ****'` → `'CO657078'`.

In [ ]:
def ex_028(D):
    return D["strings"]["account_id"].str.replace(r":\s*\*+$", "", regex=True)

ans = ex_028(D)
check(28, ans)
ans

### Exercise 29. Normalise a header

Write `norm(name)`: lowercase, then keep only letters and digits.
`'Ticket Id'` → `'ticketid'`, `'Total PnL'` → `'totalpnl'`, `'Amount ($)'` → `'amount'`.

In [ ]:
def ref_norm(name):
    return re.sub(r"[^a-z0-9]", "", str(name).lower())


check(29, ref_norm)

### Exercise 30. Clean dialer names

Write `clean_dialer_name(s)` for a Series: remove `'(...)'`, everything from `@`, everything from `;`,
collapse repeated spaces, strip, lower-case. `'Riya.S  (Ext 22)'` → `'riya.s'`.

In [ ]:
def ref_clean_dialer_name(s):
    s = s.astype("string").str.replace(r"\s*\([^)]*\)|@.*|;.*", "", regex=True)
    return (s.str.replace(r"\s+", " ", regex=True).str.strip()
             .str.replace(r"@.*", "", regex=True).str.replace(r"\(.*", "", regex=True)
             .str.strip().str.lower())


check(30, ref_clean_dialer_name)

### Exercise 31. Find damaged times

`ans` = how many values in `S['damaged_time']` are only minutes:seconds, like `'59:48.4'`
(use `.str.fullmatch`).

In [ ]:
def ex_031(D):
    return int(D["strings"]["damaged_time"].astype(str).str.fullmatch(DAMAGED).sum())

ans = ex_031(D)
check(31, ans)
ans

### Exercise 32. Blank them, then parse

Turn the damaged values into blanks with `.where`, then parse the rest with `format='ISO8601'`.
`ans` = the parsed Series.

In [ ]:
def ex_032(D):
    t = D["strings"]["damaged_time"].astype(str).str.strip()
    return pd.to_datetime(t.where(~t.str.fullmatch(DAMAGED)), format="ISO8601", errors="coerce")

ans = ex_032(D)
check(32, ans)
ans

## Combining and cleaning

### Exercise 33. Combine with a source column

Combine every Stringee export and add a `source_file` column. `ans` = the shape `(rows, columns)`.

In [ ]:
def ex_033(D):
    return _stringee(D, source=True).shape

ans = ex_033(D)
check(33, ans)
ans

### Exercise 34. Overlapping exports

On the combined Stringee rows, floor `Start time` to the second and drop duplicates on
`Account`, `Customer number`, `Start time`. `ans` = how many rows were removed.

In [ ]:
def ex_034(D):
    st = _stringee(D)
    st["Start time"] = st["Start time"].dt.floor("s")
    return len(st) - len(st.drop_duplicates(subset=["Account", "Customer number", "Start time"]))

ans = ex_034(D)
check(34, ans)
ans

### Exercise 35. Find duplicate column names

`ans` = a sorted list of the column names that appear more than once in `D['dup_cols']`.

In [ ]:
def ex_035(D):
    cols = D["dup_cols"].columns
    return sorted(set(cols[cols.duplicated()]))

ans = ex_035(D)
check(35, ans)
ans

### Exercise 36. Keep the first copy

`ans` = `D['dup_cols']` keeping only the first copy of each column name.

In [ ]:
def ex_036(D):
    df = D["dup_cols"]
    return df.loc[:, ~df.columns.duplicated()]

ans = ex_036(D)
check(36, ans)
ans

### Exercise 37. Line up two sets of headers

Map each name in `D['ark_columns']` to one in `D['main_columns']`: match on `norm()`, plus
`MANUAL = {'Account': 'accountId', 'Open Com': 'openCommission', 'Close Com': 'closeCommission'}`.
`ans` = a dict `{ark name: main name}` for the columns that match.

In [ ]:
def ex_037(D):
    return _align_map(D["ark_columns"], D["main_columns"])

ans = ex_037(D)
check(37, ans)
ans

### Exercise 38. Catch clashes

`D['clash_columns']` are ark headers; map them to `D['main_columns']` the same way.
`ans` = a sorted list of the ark names that would end up with the **same** target name.

In [ ]:
def ex_038(D):
    m = pd.Series(_align_map(D["clash_columns"], D["main_columns"]))
    return sorted(m[m.duplicated(keep=False)].index)

ans = ex_038(D)
check(38, ans)
ans

### Exercise 39. Reindex to the target columns

Rename `D['ark']` with your exercise-37 mapping (use the real `D['ark'].columns`), then
`.reindex(columns=...)` to `D['main_columns']` without `'id'`. `ans` = the resulting DataFrame.

In [ ]:
def ex_039(D):
    ark = D["ark"]
    m = _align_map(ark.columns, D["main_columns"])
    return ark[list(m)].rename(columns=m).reindex(columns=[c for c in D["main_columns"] if c != "id"])

ans = ex_039(D)
check(39, ans)
ans

### Exercise 40. Overlapping Uno exports

Combine all three Uno CSVs and drop duplicate `Call Id`s. `ans` = `(rows before, rows after)`.

In [ ]:
def ex_040(D):
    uno = _uno(D, ["uno_2026-10-28.csv", "uno_2026-10-29.csv", "uno_overlap.csv"])
    return (len(uno), len(uno.drop_duplicates(subset="Call Id")))

ans = ex_040(D)
check(40, ans)
ans

### Exercise 41. Drop one end code safely

On the combined Stringee rows, drop `CAN_NOT_MAKE_CALL` (rows with a blank code must stay).
`ans` = rows left.

In [ ]:
def ex_041(D):
    st = _stringee(D)
    return int((~st["End of Call code"].isin(["CAN_NOT_MAKE_CALL"])).sum())

ans = ex_041(D)
check(41, ans)
ans

### Exercise 42. Calls that have an agent

On the combined Stringee rows, keep rows whose `Account` is not blank and not `'---'`. `ans` = rows left.

In [ ]:
def ex_042(D):
    st = _stringee(D)
    return int((st["Account"].notna() & st["Account"].ne("---")).sum())

ans = ex_042(D)
check(42, ans)
ans

## Lookups and joins

### Exercise 43. Latest balance per client

From `D['ledger']`: `ans` = a DataFrame `ClientID`, `Balance` with each client's balance on their latest
`CreatedAt` (sort by date, newest first, keep the first row per client).

In [ ]:
def ex_043(D):
    return _latest_balance(D)

ans = ex_043(D)
check(43, ans)
ans

### Exercise 44. Same result with idxmax

Get the same answer using `D['ledger'].groupby('ClientID')['CreatedAt'].idxmax()` and `.loc`.
`ans` = DataFrame `ClientID`, `Balance`.

In [ ]:
def ex_044(D):
    led = D["ledger"]
    return led.loc[led.groupby("ClientID")["CreatedAt"].idxmax(), ["ClientID", "Balance"]].reset_index(drop=True)

ans = ex_044(D)
check(44, ans)
ans

### Exercise 45. Last trade date

`ans` = a DataFrame `ClientID`, `last_trade`: the latest `CreatedAt` among `BILL` rows.

In [ ]:
def ex_045(D):
    bills = D["ledger"][D["ledger"]["LedgerType"] == "BILL"]
    return bills.groupby("ClientID", as_index=False)["CreatedAt"].max().rename(columns={"CreatedAt": "last_trade"})

ans = ex_045(D)
check(45, ans)
ans

### Exercise 46. Fill a column with .map()

Strip spaces from `D['traded_ids']['Zynixx Id']`, then map each id to its latest balance.
`ans` = the Series of balances in the same order (NaN where not found).

In [ ]:
def ex_046(D):
    lookup = _latest_balance(D).set_index("ClientID")["Balance"]
    return D["traded_ids"]["Zynixx Id"].str.strip().map(lookup)

ans = ex_046(D)
check(46, ans)
ans

### Exercise 47. Who got no balance

`ans` = a sorted list of stripped ids from exercise 46 with no balance, excluding `'No New Id'`.

In [ ]:
def ex_047(D):
    ids = D["traded_ids"]["Zynixx Id"].str.strip()
    bal = ex_046(D)
    return sorted(ids[bal.isna() & ids.ne("No New Id")].tolist())

ans = ex_047(D)
check(47, ans)
ans

### Exercise 48. Let validate catch repeats

Run `pd.merge(left, D['ledger'][['ClientID', 'Balance']], on='ClientID', validate='many_to_one')` where
`left = D['traded_ids'].assign(ClientID=D['traded_ids']['Zynixx Id'].str.strip())`, inside `try/except`.
`ans` = the exception's class name.

In [ ]:
def ex_048(D):
    left = D["traded_ids"].assign(ClientID=D["traded_ids"]["Zynixx Id"].str.strip())
    try:
        pd.merge(left, D["ledger"][["ClientID", "Balance"]], on="ClientID", validate="many_to_one")
    except Exception as e:
        return type(e).__name__
    return "no error"

ans = ex_048(D)
check(48, ans)
ans

### Exercise 49. Spot a fan-out

Clean `D['team_old']['Dialer Name']` with your `clean_dialer_name`, then left-merge
`D['calls'][['Dialer Name']]` with it on `Dialer Name`. `ans` = `(rows before, rows after)`.

In [ ]:
def ex_049(D):
    team = D["team_old"].copy()
    team["Dialer Name"] = ref_clean_dialer_name(team["Dialer Name"]).astype(str)
    left = D["calls"][["Dialer Name"]]
    return (len(left), len(left.merge(team, on="Dialer Name", how="left")))

ans = ex_049(D)
check(49, ans)
ans

### Exercise 50. Names shared by two people

In `D['team_old']`: drop rows whose `Email` contains `'inactive'` (any case), clean the names,
drop duplicate (`Dialer Name`, `Email`). `ans` = a sorted list of names still linked to more than one email.

In [ ]:
def ex_050(D):
    t = D["team_old"]
    t = t[~t["Email"].str.contains("inactive", case=False, na=False)].copy()
    t["Dialer Name"] = ref_clean_dialer_name(t["Dialer Name"]).astype(str)
    t = t.drop_duplicates(subset=["Dialer Name", "Email"])
    return sorted(t.loc[t["Dialer Name"].duplicated(keep=False), "Dialer Name"].unique())

ans = ex_050(D)
check(50, ans)
ans

### Exercise 51. Unmatched calls per source

Left-merge `D['calls_all']` (cleaned calls) with `D['team']` (cleaned team list) on `Dialer Name`.
`ans` = a dict `{Source: calls with no CRM ID}`.

In [ ]:
def ex_051(D):
    m = D["calls_all"].merge(D["team"][["Dialer Name", "CRM ID"]], on="Dialer Name", how="left")
    return m[m["CRM ID"].isna()].groupby("Source").size().to_dict()

ans = ex_051(D)
check(51, ans)
ans

### Exercise 52. The group's value on every row

`ans` = a Series: for every ledger row, that client's latest `CreatedAt` (use `transform('max')`).

In [ ]:
def ex_052(D):
    return D["ledger"].groupby("ClientID")["CreatedAt"].transform("max")

ans = ex_052(D)
check(52, ans)
ans

## Group calculations

### Exercise 53. Calls per agent per day

From `D['calls']`: `ans` = a DataFrame `CRM ID`, `Date`, `calls`.

In [ ]:
def ex_053(D):
    return D["calls"].groupby(["CRM ID", "Date"]).size().reset_index(name="calls")

ans = ex_053(D)
check(53, ans)
ans

### Exercise 54. Named aggregation

Per `CRM ID` and `Date`: `dialed` (rows), `connected` (Call Status == 'connected'),
`unique_numbers` (distinct Number). `ans` = a DataFrame with those 5 columns.

In [ ]:
def ex_054(D):
    c = D["calls"].assign(is_conn=D["calls"]["Call Status"].eq("connected").astype(int))
    return c.groupby(["CRM ID", "Date"]).agg(dialed=("Call Status", "count"), connected=("is_conn", "sum"),
                                             unique_numbers=("Number", "nunique")).reset_index()

ans = ex_054(D)
check(54, ans)
ans

### Exercise 55. Talk time of connected calls only

`ans` = a DataFrame `CRM ID`, `Date`, `talk_s`: whole seconds of `Talk Time`, counting connected calls
only (`.where(connected, 0)` before summing).

In [ ]:
def ex_055(D):
    c = D["calls"].assign(talk_s=_talk_s(D).where(D["calls"]["Call Status"].eq("connected"), 0))
    return c.groupby(["CRM ID", "Date"], as_index=False)["talk_s"].sum()

ans = ex_055(D)
check(55, ans)
ans

### Exercise 56. Long connected calls

`ans` = a DataFrame `CRM ID`, `Date`, `gt30`: number of connected calls with talk time over 30 seconds.

In [ ]:
def ex_056(D):
    c = D["calls"].assign(gt30=(D["calls"]["Call Status"].eq("connected") & (_talk_s(D) > 30)).astype(int))
    return c.groupby(["CRM ID", "Date"], as_index=False)["gt30"].sum()

ans = ex_056(D)
check(56, ans)
ans

### Exercise 57. The previous call's start

`D['calls']` is sorted by Date, CRM ID, Call Start Time. `ans` = a Series of the previous call's start
for the same CRM ID and Date (`groupby(...).shift()`; NaT for each first call).

In [ ]:
def ex_057(D):
    return _gap_parts(D)[1]

ans = ex_057(D)
check(57, ans)
ans

### Exercise 58. Gap in seconds

gap = start − (previous start + previous `Total Call Duration`); negatives → 0, missing → 0.
`ans` = a Series of whole seconds (int).

In [ ]:
def ex_058(D):
    return _gap_seconds(D)

ans = ex_058(D)
check(58, ans)
ans

### Exercise 59. Flagged gaps

Count gaps over 60 seconds where **both** this call and the previous call start between 09:30 and 18:30
(inclusive, time of day). `ans` = an int.

In [ ]:
def ex_059(D):
    calls, prev_start, gap = _gap_parts(D)
    tod = calls["Call Start Time"] - calls["Call Start Time"].dt.normalize()
    ptod = prev_start - prev_start.dt.normalize()
    lo, hi = pd.Timedelta("09:30:00"), pd.Timedelta("18:30:00")
    inside = tod.between(lo, hi) & ptod.between(lo, hi)
    return int((inside & (gap.dt.total_seconds() > 60).fillna(False)).sum())

ans = ex_059(D)
check(59, ans)
ans

### Exercise 60. Break allowance

Sum the gap seconds per `CRM ID` and `Date`, then subtract 3600 where the total is over 3600.
`ans` = a DataFrame `CRM ID`, `Date`, `gap_s`.

In [ ]:
def ex_060(D):
    return _gap_table(D)

ans = ex_060(D)
check(60, ans)
ans

### Exercise 61. Login seconds

`login_s` = total call seconds (sum of whole-second `Total Call Duration`) + the adjusted gap from
exercise 60 + 3600. `ans` = a DataFrame `CRM ID`, `Date`, `login_s`.

In [ ]:
def ex_061(D):
    return _login_table(D)

ans = ex_061(D)
check(61, ans)
ans

### Exercise 62. Attendance bands with np.select

From exercise 61: under 4h30 → `'Absent'`, under 6h → `'Half Day'`, under 8h30 → `'Warning'`,
otherwise `'Present'`. `ans` = a dict `{label: count}`.

In [ ]:
def ex_062(D):
    s = _login_table(D)["login_s"]
    label = np.select([s < 4.5 * 3600, s < 6 * 3600, s < 8.5 * 3600], ["Absent", "Half Day", "Warning"], default="Present")
    return pd.Series(label).value_counts().to_dict()

ans = ex_062(D)
check(62, ans)
ans

### Exercise 63. Hour slots

Label each call's start hour with `D['SLOT_LABELS']`: hour < 8 → `'0 to 8 AM'`, 8 → `'8 to 9 AM'`, …,
19 → `'19 to 20 PM'`, 20 or later → `'20+'` (hint: `(hour - 7).clip(0, 13)`). `ans` = a dict `{slot: calls}`.

In [ ]:
def ex_063(D):
    return _slots(D).value_counts().to_dict()

ans = ex_063(D)
check(63, ans)
ans

### Exercise 64. Calls per slot

`ans` = a pivot DataFrame (after `reset_index()`): columns `CRM ID`, `Date`, then one column per slot
that appears (calls, 0 if none).

In [ ]:
def ex_064(D):
    c = D["calls"].assign(Interval=_slots(D))
    p = c.pivot_table(index=["CRM ID", "Date"], columns="Interval", values="Call Status", aggfunc="count", fill_value=0)
    p.columns.name = None
    return p.reset_index()

ans = ex_064(D)
check(64, ans)
ans

### Exercise 65. Trading days per month

From `BILL` rows in `D['ledger']`: `ans` = a pivot DataFrame (after `reset_index()`) with `ClientID` and one
column per month (`'2026-08'` text) holding the number of **distinct dates** traded.

In [ ]:
def ex_065(D):
    return _bill_days_pivot(D).reset_index()

ans = ex_065(D)
check(65, ans)
ans

### Exercise 66. Check for the impossible

`ans` = the largest number in your exercise-65 month columns (it must be 31 or less).

In [ ]:
def ex_066(D):
    return int(_bill_days_pivot(D).max().max())

ans = ex_066(D)
check(66, ans)
ans

### Exercise 67. Total and average

From exercise 65, add `Total Trading Days` (sum of month columns) and `Average` (their mean, 2 decimals).
`ans` = a DataFrame `ClientID`, `Total Trading Days`, `Average`.

In [ ]:
def ex_067(D):
    p = _bill_days_pivot(D)
    months = list(p.columns)
    p["Total Trading Days"] = p[months].sum(axis=1)
    p["Average"] = p[months].mean(axis=1).round(2)
    return p.reset_index()[["ClientID", "Total Trading Days", "Average"]]

ans = ex_067(D)
check(67, ans)
ans

### Exercise 68. Confidence tiers

`D['tiers']` has `last_txn`, `call_1_time` (latest missed call), `call_5_time` (first of the streak) and
`call_days`. Low if `call_days == 1` or `last_txn <= call_5_time`; else High if `last_txn > call_1_time`;
else Medium. `ans` = a dict `{tier: count}`.

In [ ]:
def ex_068(D):
    t = D["tiers"]
    low = (t["call_days"] == 1) | (t["last_txn"] <= t["call_5_time"])
    high = t["last_txn"] > t["call_1_time"]
    return pd.Series(np.select([low, high], ["Low", "High"], default="Medium")).value_counts().to_dict()

ans = ex_068(D)
check(68, ans)
ans

## Dates and times

### Exercise 69. Yesterday

With `today = D['TODAY']`, `ans` = the number of ledger rows from yesterday (`>=` start, `<` end).

In [ ]:
def ex_069(D):
    today = D["TODAY"].normalize()
    c = D["ledger"]["CreatedAt"]
    return int(((c >= today - pd.Timedelta(days=1)) & (c < today)).sum())

ans = ex_069(D)
check(69, ans)
ans

### Exercise 70. The last 7 full days

`ans` = ledger rows from the 7 days before `D['TODAY']`, not including `D['TODAY']`.

In [ ]:
def ex_070(D):
    today = D["TODAY"].normalize()
    c = D["ledger"]["CreatedAt"]
    return int(((c >= today - pd.Timedelta(days=7)) & (c < today)).sum())

ans = ex_070(D)
check(70, ans)
ans

### Exercise 71. Month so far

`ans` = ledger rows from the 1st of `D['TODAY']`'s month up to, but not including, `D['TODAY']`.

In [ ]:
def ex_071(D):
    today = D["TODAY"].normalize()
    c = D["ledger"]["CreatedAt"]
    return int(((c >= today.replace(day=1)) & (c < today)).sum())

ans = ex_071(D)
check(71, ans)
ans

### Exercise 72. Days the calls were spread over

`D['last5']` lists each lead's last calls (`lead_id`, `rn`, `call_time`). `ans` = a DataFrame `lead_id`,
`call_days`: the number of distinct dates among `rn` 1 to 5.

In [ ]:
def ex_072(D):
    t = D["last5"]
    return (t[t["rn"] <= 5].assign(d=t["call_time"].dt.normalize())
             .groupby("lead_id", as_index=False)["d"].nunique().rename(columns={"d": "call_days"}))

ans = ex_072(D)
check(72, ans)
ans

### Exercise 73. UTC to Dubai

Treat ledger `CreatedAt` as UTC, convert it to `'Asia/Dubai'` and drop the time zone.
`ans` = the converted Series.

In [ ]:
def ex_073(D):
    return D["ledger"]["CreatedAt"].dt.tz_localize("UTC").dt.tz_convert("Asia/Dubai").dt.tz_localize(None)

ans = ex_073(D)
check(73, ans)
ans

### Exercise 74. Inside working hours

`ans` = the number of calls in `D['calls']` starting between 09:30 and 18:30, inclusive.

In [ ]:
def ex_074(D):
    t = D["calls"]["Call Start Time"]
    tod = t - t.dt.normalize()
    return int(tod.between(pd.Timedelta("09:30:00"), pd.Timedelta("18:30:00")).sum())

ans = ex_074(D)
check(74, ans)
ans

### Exercise 75. Calls per day

`ans` = a dict `{'YYYY-MM-DD': calls}` for `D['calls']`.

In [ ]:
def ex_075(D):
    return D["calls"].groupby(D["calls"]["Date"].dt.strftime("%Y-%m-%d")).size().to_dict()

ans = ex_075(D)
check(75, ans)
ans

### Exercise 76. Missing days

For the client `D['gap_client']`, look at `BILL` rows from 1 Oct 2026 onwards. `ans` = a sorted list of dates
(`'YYYY-MM-DD'`) between their first and last October BILL date on which they had **no** BILL
(hint: `pd.date_range(...).difference(...)`).

In [ ]:
def ex_076(D):
    led = D["ledger"]
    b = led[(led["ClientID"] == D["gap_client"]) & (led["LedgerType"] == "BILL") & (led["CreatedAt"] >= "2026-10-01")]
    days = b["CreatedAt"].dt.normalize()
    full = pd.date_range(days.min(), days.max(), freq="D")
    return [d.strftime("%Y-%m-%d") for d in full.difference(pd.DatetimeIndex(days.unique()))]

ans = ex_076(D)
check(76, ans)
ans

### Exercise 77. Rows per month

`ans` = a dict `{'2026-08': rows, ...}` of ledger rows per month (`dt.to_period('M')`).

In [ ]:
def ex_077(D):
    return D["ledger"]["CreatedAt"].dt.to_period("M").astype(str).value_counts().to_dict()

ans = ex_077(D)
check(77, ans)
ans

### Exercise 78. Seconds to HH:MM:SS

Write `hms(seconds)` turning a Series of whole seconds into `'HH:MM:SS'` text (hours may go past 24).

In [ ]:
def ref_hms(seconds):
    s = pd.Series(seconds).fillna(0).astype("int64")
    h, m, sec = s // 3600, (s % 3600) // 60, s % 60
    return h.astype(str).str.zfill(2) + ":" + m.astype(str).str.zfill(2) + ":" + sec.astype(str).str.zfill(2)


check(78, ref_hms)

## Checks and output

### Exercise 79. Two reports must agree

Count calls per `CRM ID` and `Date` two ways: `groupby().size()`, and the row sum of the slot columns of
your exercise-64 pivot. `ans` = how many rows differ (should be 0).

In [ ]:
def ex_079(D):
    a = D["calls"].groupby(["CRM ID", "Date"]).size().rename("a").reset_index()
    p = ex_064(D)
    p["b"] = p.drop(columns=["CRM ID", "Date"]).sum(axis=1)
    m = a.merge(p[["CRM ID", "Date", "b"]], on=["CRM ID", "Date"], how="outer")
    return int((m["a"] != m["b"]).sum())

ans = ex_079(D)
check(79, ans)
ans

### Exercise 80. A check that stops the run

Write `require_parsed(s, label)`: return `s` unchanged, or raise `ValueError` mentioning how many values are
missing (NaT/NaN).

In [ ]:
def _ref_require_parsed(s, label):
    bad = int(s.isna().sum())
    if bad:
        raise ValueError(f"{bad} {label} values could not be read")
    return s


check(80, _ref_require_parsed)

### Exercise 81. Durations Excel can add up

Write `D['calls'][['CRM ID', 'Talk Time']].head(20)` to `P['out_dir'] / 'durations.xlsx'`, with
`Talk Time` as a fraction of a day (`seconds / 86400`) and the number format `[h]:mm:ss` on that column.
Read cell B2 back with `openpyxl.load_workbook`. `ans` = its `number_format`.

In [ ]:
def ex_081(D):
    return "[h]:mm:ss"

ans = ex_081(D)
check(81, ans)
ans

### Exercise 82. Add a sheet to a workbook

Copy `team_2026-10-30.xlsx` to `P['out_dir'] / 'team_copy.xlsx'`, then add a sheet `result` with
`D['calls'].head()` using `ExcelWriter(..., mode='a', if_sheet_exists='replace')`. Run it twice.
`ans` = the workbook's sheet names (list).

In [ ]:
def ex_082(D):
    return ["Sheet1", "result"]

ans = ex_082(D)
check(82, ans)
ans

### Exercise 83. Excel's fraction of a day

`ans` = 5 minutes 40 seconds as a fraction of a day, rounded to 6 decimals.

In [ ]:
def ex_083(D):
    return round(340 / 86400, 6)

ans = ex_083(D)
check(83, ans)
ans

### Exercise 84. Rows versus distinct keys

Left-merge `D['customer_status_history']` onto `D['leads']` (`leads.customer_id` = `customer_id`).
`ans` = `(rows after the merge, distinct lead ids)`. Different numbers mean every lead repeats.

In [ ]:
def ex_084(D):
    m = D["leads"].merge(D["customer_status_history"], on="customer_id", how="left")
    return (len(m), int(m["id_x"].nunique()))

ans = ex_084(D)
check(84, ans)
ans

### Exercise 85. Rows dropped at each step

From the combined Stringee rows: (1) drop `CAN_NOT_MAKE_CALL`; (2) floor `Start time` to the second and drop
duplicates on Account, Customer number, Start time; (3) drop blank or `'---'` Account.
`ans` = a dict `{'end_code': n1, 'duplicates': n2, 'no_agent': n3}` of rows removed at each step.

In [ ]:
def ex_085(D):
    st = _stringee(D)
    n0 = len(st)
    st = st[~st["End of Call code"].isin(["CAN_NOT_MAKE_CALL"])]
    n1 = len(st)
    st = st.assign(**{"Start time": st["Start time"].dt.floor("s")}).drop_duplicates(
        subset=["Account", "Customer number", "Start time"])
    n2 = len(st)
    st = st[st["Account"].notna() & st["Account"].ne("---")]
    return {"end_code": n0 - n1, "duplicates": n1 - n2, "no_agent": n2 - len(st)}

ans = ex_085(D)
check(85, ans)
ans

### Exercise 86. Environment variables are read by name

In your cell, set `os.environ['PRACTICE_TOKEN'] = 'abc123'`, then read it back **by its name**.
`ans` = the length of the value you read.

In [ ]:
def ex_086(D):
    return 6

ans = ex_086(D)
check(86, ans)
ans

## Polars

### Exercise 87. A lazy plan

`lf = pl.concat([pl.scan_csv(f) for f in sorted(P['tx_dir'].glob('*.csv'))], how='diagonal_relaxed')`.
`ans` = the column names (list), without reading the data (`collect_schema()`).

In [ ]:
def ex_087(D):
    return _lf(D).collect_schema().names()

ans = ex_087(D)
check(87, ans)
ans

### Exercise 88. Count rows lazily

`ans` = the total number of rows, from `lf.select(pl.len()).collect().item()`.

In [ ]:
def ex_088(D):
    import polars as pl
    return int(_lf(D).select(pl.len()).collect().item())

ans = ex_088(D)
check(88, ans)
ans

### Exercise 89. Filter before collecting

`ans` = the number of rows whose `accountId` is in `D['tx_accounts']` and `action == 'New'`,
using `.filter(...)` before `.collect()`.

In [ ]:
def ex_089(D):
    import polars as pl
    return int(_lf(D).filter(pl.col("accountId").is_in(D["tx_accounts"]) & (pl.col("action") == "New"))
               .select(pl.len()).collect().item())

ans = ex_089(D)
check(89, ans)
ans

### Exercise 90. Group in Polars

`ans` = a dict `{script: rows}` using `lf.group_by('script').agg(pl.len())`.

In [ ]:
def ex_090(D):
    import polars as pl
    df = _lf(D).group_by("script").agg(pl.len().alias("n")).collect()
    return dict(zip(df["script"].to_list(), df["n"].to_list()))

ans = ex_090(D)
check(90, ans)
ans

### Exercise 91. Tag each file

Scan each transactions CSV with `.with_columns(pl.lit(f.name).alias('source_file'))`, concat, then count.
`ans` = a dict `{file name: rows}`.

In [ ]:
def ex_091(D):
    import polars as pl
    lf = pl.concat([pl.scan_csv(f).with_columns(pl.lit(f.name).alias("source_file"))
                    for f in sorted(D["paths"]["tx_dir"].glob("*.csv"))], how="diagonal_relaxed")
    df = lf.group_by("source_file").agg(pl.len().alias("n")).collect()
    return dict(zip(df["source_file"].to_list(), df["n"].to_list()))

ans = ex_091(D)
check(91, ans)
ans

### Exercise 92. When headers differ

Scan `P['tx_dir'] / 'tx_1.csv'` and `P['ark_dir'] / 'ark_1.csv'` and concat with `how='diagonal_relaxed'`.
`ans` = the number of columns (different names don't line up, so you get both sets).

In [ ]:
def ex_092(D):
    import polars as pl
    lf = pl.concat([pl.scan_csv(D["paths"]["tx_dir"] / "tx_1.csv"), pl.scan_csv(D["paths"]["ark_dir"] / "ark_1.csv")],
                   how="diagonal_relaxed")
    return len(lf.collect_schema().names())

ans = ex_092(D)
check(92, ans)
ans

### Exercise 93. Sink to Parquet

Write `lf` to `P['out_dir'] / 'tx.parquet'` with `lf.sink_parquet(...)`, then count its rows with
`pl.scan_parquet(...).select(pl.len()).collect().item()`. `ans` = that count.

In [ ]:
def ex_093(D):
    return ex_088(D)

ans = ex_093(D)
check(93, ans)
ans

### Exercise 94. Back to pandas, small

`ans` = a **pandas** DataFrame with `ticketId` and `amount` for `script == 'XAUUSD'`
(`select` + `filter`, `.collect()`, then `.to_pandas()`).

In [ ]:
def ex_094(D):
    import polars as pl
    return (_lf(D).filter(pl.col("script") == "XAUUSD").select(["ticketId", "amount"]).collect().to_pandas())

ans = ex_094(D)
check(94, ans)
ans

## SQL

### Exercise 95. Newest call per lead

`ans = sql(...)`: `lead_id`, `started_at` of each lead's newest call, ignoring `status` `'placing'` and
`'call_failed'`, using `ROW_NUMBER()` (calls without a disposition count here).

In [ ]:
def ex_095(D):
    return _sql("""
        SELECT lead_id, started_at FROM (
            SELECT lead_id, started_at,
                   ROW_NUMBER() OVER (PARTITION BY lead_id ORDER BY started_at DESC, id DESC) AS rn
            FROM call_logs WHERE status NOT IN ('placing', 'call_failed')) t
        WHERE rn = 1""")

ans = ex_095(D)
check(95, ans)
ans

### Exercise 96. Last 3 all 'No Answer'

Rank each lead's **dispositioned** calls newest first (join `lead_calls` and `lead_call_dispositions`; skip
placing/call_failed). `ans` = a sorted list of lead ids whose last 3 are all `'No Answer'`
(`BOOL_AND` + `HAVING COUNT(*) = 3`).

In [ ]:
def ex_096(D):
    df = _sql(RANKED + """
        SELECT lead_id FROM ranked WHERE rn <= 3 GROUP BY lead_id
        HAVING COUNT(*) = 3 AND BOOL_AND(disposition = 'No Answer') ORDER BY lead_id""")
    return df["lead_id"].tolist()

ans = ex_096(D)
check(96, ans)
ans

### Exercise 97. Same leads with FILTER

Same answer using `COUNT(*) FILTER (WHERE disposition = 'No Answer') = 3` instead of `BOOL_AND`.

In [ ]:
def ex_097(D):
    df = _sql(RANKED + """
        SELECT lead_id FROM ranked WHERE rn <= 3 GROUP BY lead_id
        HAVING COUNT(*) FILTER (WHERE disposition = 'No Answer') = 3 ORDER BY lead_id""")
    return df["lead_id"].tolist()

ans = ex_097(D)
check(97, ans)
ans

### Exercise 98. Latest disposition with DISTINCT ON

`ans` = `lead_id`, `disposition` of each lead's newest dispositioned call, using
`SELECT DISTINCT ON (lead_id) ... ORDER BY lead_id, started_at DESC, id DESC`.

In [ ]:
def ex_098(D):
    return _sql("""
        SELECT DISTINCT ON (cl.lead_id) cl.lead_id, d.name AS disposition
        FROM call_logs cl
        JOIN lead_calls lc ON lc.call_log_id = cl.id
        JOIN lead_call_dispositions d ON d.id = lc.disposition_id
        WHERE cl.status NOT IN ('placing', 'call_failed')
        ORDER BY cl.lead_id, cl.started_at DESC, cl.id DESC""")

ans = ex_098(D)
check(98, ans)
ans

### Exercise 99. A window aggregate

On the ranked rows with `rn <= 5`, add `COUNT(*) OVER (PARTITION BY lead_id)`.
`ans` = the number of leads that have all 5.

In [ ]:
def ex_099(D):
    df = _sql(RANKED + """
        SELECT COUNT(DISTINCT lead_id) AS n FROM (
            SELECT lead_id, COUNT(*) OVER (PARTITION BY lead_id) AS c FROM ranked WHERE rn <= 5) t
        WHERE c = 5""")
    return int(df["n"].iloc[0])

ans = ex_099(D)
check(99, ans)
ans

### Exercise 100. A sequence as text

`ans` = `lead_id`, `last_3`: the last 3 dispositions, newest first, joined with `' -> '`
(`STRING_AGG(... ORDER BY rn)`).

In [ ]:
def ex_100(D):
    return _sql(RANKED + """
        SELECT lead_id, STRING_AGG(disposition, ' -> ' ORDER BY rn) AS last_3
        FROM ranked WHERE rn <= 3 GROUP BY lead_id""")

ans = ex_100(D)
check(100, ans)
ans

### Exercise 101. Two-level breakdown

Over each lead's last 5 calls: count per disposition, then `STRING_AGG(disposition || ' x' || cnt, ', '
ORDER BY cnt DESC, disposition)`. `ans` = `lead_id`, `breakdown`, e.g. `'No Answer x3, Zero x2'`.

In [ ]:
def ex_101(D):
    return _sql(RANKED + """
        , counts AS (SELECT lead_id, disposition, COUNT(*) AS cnt FROM ranked WHERE rn <= 5 GROUP BY 1, 2)
        SELECT lead_id, STRING_AGG(disposition || ' x' || cnt, ', ' ORDER BY cnt DESC, disposition) AS breakdown
        FROM counts GROUP BY lead_id""")

ans = ex_101(D)
check(101, ans)
ans

### Exercise 102. Match names after normalising

Put the 8 names in `D['UNREACHABLE']` in a `VALUES` list. Normalise both sides with
`regexp_replace(lower(x), '[^a-z0-9]', '', 'g')`. `ans` = how many dispositioned calls are unreachable.

In [ ]:
def ex_102(D):
    vals = ", ".join("('" + n.replace("'", "''") + "')" for n in D["UNREACHABLE"])
    df = _sql(RANKED + f"""
        , targets AS (SELECT regexp_replace(lower(name), '[^a-z0-9]', '', 'g') AS k FROM (VALUES {vals}) v(name))
        SELECT COUNT(*) AS n FROM ranked
        WHERE regexp_replace(lower(disposition), '[^a-z0-9]', '', 'g') IN (SELECT k FROM targets)""")
    return int(df["n"].iloc[0])

ans = ex_102(D)
check(102, ans)
ans

### Exercise 103. Rows into columns

`ans` = `lead_id`, `call_1`, `call_2`, `call_3` (dispositions, newest first) using
`MAX(disposition) FILTER (WHERE rn = 1)` and so on.

In [ ]:
def ex_103(D):
    return _sql(RANKED + """
        SELECT lead_id,
               MAX(disposition) FILTER (WHERE rn = 1) AS call_1,
               MAX(disposition) FILTER (WHERE rn = 2) AS call_2,
               MAX(disposition) FILTER (WHERE rn = 3) AS call_3
        FROM ranked GROUP BY lead_id""")

ans = ex_103(D)
check(103, ans)
ans

### Exercise 104. De-duplicate with DISTINCT ON

From `customer_status_history`, keep one row per (`customer_id`, `occurred_at`): the one with the latest
`created_at`. `ans` = the number of rows kept.

In [ ]:
def ex_104(D):
    return int(_sql(DEDUP + "SELECT COUNT(*) AS n FROM dedup")["n"].iloc[0])

ans = ex_104(D)
check(104, ans)
ans

### Exercise 105. Current status

On the de-duplicated rows: `ans` = `customer_id`, `rt_status` using
`(ARRAY_AGG(new_status ORDER BY occurred_at DESC))[1]`.

In [ ]:
def ex_105(D):
    return _sql(DEDUP + """
        SELECT customer_id, (ARRAY_AGG(new_status ORDER BY occurred_at DESC))[1] AS rt_status
        FROM dedup GROUP BY customer_id""")

ans = ex_105(D)
check(105, ans)
ans

### Exercise 106. Status journey

On the de-duplicated rows: `ans` = `customer_id`, `journey` = statuses oldest first joined with `' -> '`.

In [ ]:
def ex_106(D):
    return _sql(DEDUP + """
        SELECT customer_id, STRING_AGG(new_status, ' -> ' ORDER BY occurred_at) AS journey
        FROM dedup GROUP BY customer_id""")

ans = ex_106(D)
check(106, ans)
ans

### Exercise 107. Fan-out in SQL

`ans` = a tuple of two row counts: `leads LEFT JOIN customer_status_history` directly, and
`leads LEFT JOIN` a one-row-per-customer summary of it.

In [ ]:
def ex_107(D):
    a = _sql("SELECT COUNT(*) AS n FROM leads l LEFT JOIN customer_status_history h ON h.customer_id = l.customer_id")
    b = _sql("""SELECT COUNT(*) AS n FROM leads l LEFT JOIN
                (SELECT customer_id, COUNT(*) AS events FROM customer_status_history GROUP BY customer_id) s
                ON s.customer_id = l.customer_id""")
    return (int(a["n"].iloc[0]), int(b["n"].iloc[0]))

ans = ex_107(D)
check(107, ans)
ans

### Exercise 108. A half-open date range

`ans` = the number of `call_logs` rows that started on 2026-10-29
(`started_at >= '2026-10-29' AND started_at < '2026-10-30'`).

In [ ]:
def ex_108(D):
    return int(_sql("""SELECT COUNT(*) AS n FROM call_logs
                       WHERE started_at >= TIMESTAMP '2026-10-29' AND started_at < TIMESTAMP '2026-10-30'""")["n"].iloc[0])

ans = ex_108(D)
check(108, ans)
ans

### Exercise 109. Calls with no disposition

`ans` = the number of `completed` calls with no row in `lead_calls` (`LEFT JOIN ... WHERE lc.call_log_id IS NULL`).

In [ ]:
def ex_109(D):
    return int(_sql("""SELECT COUNT(*) AS n FROM call_logs cl LEFT JOIN lead_calls lc ON lc.call_log_id = cl.id
                       WHERE cl.status = 'completed' AND lc.call_log_id IS NULL""")["n"].iloc[0])

ans = ex_109(D)
check(109, ans)
ans

### Exercise 110. WHERE versus HAVING

`ans` = a sorted list of leads with at least 5 dispositioned calls, of which at least 2 are
`'Interested'` or `'Call Later'`.

In [ ]:
def ex_110(D):
    df = _sql(RANKED + """
        SELECT lead_id FROM ranked GROUP BY lead_id
        HAVING COUNT(*) >= 5 AND COUNT(*) FILTER (WHERE disposition IN ('Interested', 'Call Later')) >= 2
        ORDER BY lead_id""")
    return df["lead_id"].tolist()

ans = ex_110(D)
check(110, ans)
ans

## Capstones

### Exercise 111. The daily summary

From `D['calls']`, per `CRM ID` and `Date`: `Total_Dialed_Calls`, `Total_Connected_Calls`,
`Total_Talk_Time` (connected, whole seconds), `Total_Gap_Duration` (after the 1-hour allowance) and
`Login_Seconds` (exercise 61). `ans` = a DataFrame with those 7 columns.

In [ ]:
def ex_111(D):
    a = ex_054(D).rename(columns={"dialed": "Total_Dialed_Calls", "connected": "Total_Connected_Calls"})
    a = a.drop(columns="unique_numbers")
    a = a.merge(ex_055(D).rename(columns={"talk_s": "Total_Talk_Time"}), on=["CRM ID", "Date"])
    a = a.merge(_gap_table(D).rename(columns={"gap_s": "Total_Gap_Duration"}), on=["CRM ID", "Date"])
    a = a.merge(_login_table(D).rename(columns={"login_s": "Login_Seconds"}), on=["CRM ID", "Date"])
    return a

ans = ex_111(D)
check(111, ans)
ans

### Exercise 112. Hourly report with a built-in check

Build calls per slot per `CRM ID` and `Date` (exercise 64) plus a `Total Calls` column, and `assert` it equals
`Total_Dialed_Calls` from exercise 111. `ans` = the hourly DataFrame.

In [ ]:
def ex_112(D):
    p = ex_064(D)
    p["Total Calls"] = p.drop(columns=["CRM ID", "Date"]).sum(axis=1)
    chk = p.merge(ex_111(D), on=["CRM ID", "Date"])
    assert (chk["Total Calls"] == chk["Total_Dialed_Calls"]).all()
    return p

ans = ex_112(D)
check(112, ans)
ans

### Exercise 113. Client activity report

For the stripped ids in `D['traded_ids']` (excluding `'No New Id'`) that have BILL rows: month columns of
trading days, `Total Trading Days`, `Average` (2 decimals), latest `Balance`, and `Last Trade` (latest BILL
`CreatedAt`). `ans` = a DataFrame with `ClientID` first.

In [ ]:
def ex_113(D):
    ids = D["traded_ids"]["Zynixx Id"].str.strip()
    ids = set(ids[ids.ne("No New Id")])
    p = _bill_days_pivot(D, ids)
    months = list(p.columns)
    p["Total Trading Days"] = p[months].sum(axis=1)
    p["Average"] = p[months].mean(axis=1).round(2)
    r = p.reset_index().merge(_latest_balance(D), on="ClientID", how="left")
    return r.merge(ex_045(D).rename(columns={"last_trade": "Last Trade"}), on="ClientID", how="left")

ans = ex_113(D)
check(113, ans)
ans

### Exercise 114. Status journey in pandas

Without SQL: keep one row per (`customer_id`, `occurred_at`) with the latest `created_at`, then per customer
`rt_status` (latest `new_status`) and `journey` (oldest first, `' -> '`).
`ans` = a DataFrame `customer_id`, `rt_status`, `journey`.

In [ ]:
def ex_114(D):
    h = D["customer_status_history"].sort_values(["customer_id", "occurred_at", "created_at", "id"],
                                                  ascending=[True, True, False, False])
    h = h.drop_duplicates(subset=["customer_id", "occurred_at"], keep="first").sort_values(["customer_id", "occurred_at"])
    return h.groupby("customer_id").agg(rt_status=("new_status", "last"),
                                        journey=("new_status", lambda s: " -> ".join(s))).reset_index()

ans = ex_114(D)
check(114, ans)
ans

### Exercise 115. Unreachable leads in pandas

Without SQL: join `call_logs`, `lead_calls` and `lead_call_dispositions`, skip placing/call_failed, rank each
lead's calls newest first (`groupby().cumcount()`), and `ans` = a sorted list of leads with at least 5 calls whose
last 5 are all in `D['UNREACHABLE']` (compare with `norm()`).

In [ ]:
def ex_115(D):
    c = (D["call_logs"].merge(D["lead_calls"], left_on="id", right_on="call_log_id")
         .merge(D["lead_call_dispositions"].rename(columns={"id": "disp_id", "name": "disposition"}),
                left_on="disposition_id", right_on="disp_id"))
    c = c[~c["status"].isin(["placing", "call_failed"])]
    c = c.sort_values(["lead_id", "started_at", "id"], ascending=[True, False, False])
    c["rn"] = c.groupby("lead_id").cumcount() + 1
    keys = {ref_norm(n) for n in D["UNREACHABLE"]}
    last5 = c[c["rn"] <= 5].assign(hit=lambda x: x["disposition"].map(ref_norm).isin(keys))
    g = last5.groupby("lead_id").agg(n=("hit", "size"), all_hit=("hit", "all"))
    return sorted(g[(g["n"] == 5) & g["all_hit"]].index.tolist())

ans = ex_115(D)
check(115, ans)
ans